# Lab 5 - Głębokie sieci w PyTorch dla danych tabelarycznych i rzetelna analiza wyników

> **Wersja dla studentów.** Komórki z `# TODO` uzupełnij samodzielnie.

**Kurs:** Uczenie głębokie (Informatyka, II stopień, sem. III, studia niestacjonarne, PWr)
**Prowadząca:** dr inż. Natalia Piórkowska
**Czas:** 180 min (Google Colab, GPU T4)
**Treść programowa:** T4 - głębokie sieci w regresji i klasyfikacji w PyTorch, analiza wyników.

## Cel zajęć

Na poprzednich laboratoriach trenowaliśmy sieci w Keras na obrazach. Dziś przechodzimy do **PyTorch** i do **danych tabelarycznych** (wiersze = obiekty, kolumny = cechy), które dominują w medycynie, finansach i przemyśle. Najważniejsze przesłanie zajęć brzmi jednak inaczej niż "jak zbudować sieć":

> **Wynik modelu jest wart tyle, ile procedura, którą go zmierzono.**

Nauczysz się:
1. budować pełny potok w PyTorch: `Dataset` -> `DataLoader` -> `nn.Module` -> pętla trening/walidacja -> ręczny *early stopping* -> zapis najlepszych wag,
2. uczciwie porównywać sieć z modelami klasycznymi (regresja logistyczna, las losowy, gradient boosting),
3. analizować wyniki klasyfikacji (macierz pomyłek, ROC, PR, dobór progu, kalibracja) i regresji (MAE, RMSE, $R^2$, analiza reszt),
4. **rozpoznawać i eliminować wyciek danych (data leakage)** oraz budować poprawny pipeline,
5. szacować niepewność wyników (walidacja krzyżowa, przedziały ufności, bootstrap),
6. interpretować model (permutation importance) i stroić hiperparametry (random search) bez "podglądania" zbioru testowego.

## Efekty uczenia się (karta przedmiotu)
- **UGŁ_W01** - zna modele klasyczne ML i głębokie oraz metody ich uczenia (porównanie MLP z LogReg / RF / GBDT).
- **UGŁ_W02** - zna procesy cyklu życia systemu i zaawansowane narzędzia DL (PyTorch, pipeline, walidacja, zapis modelu).
- **UGŁ_U01** - tworzy i weryfikuje hipotezy dotyczące problemów rozwiązywanych modelami DL (np. "sieć jest lepsza od regresji logistycznej" - z przedziałem niepewności).
- **UGŁ_U02** - proponuje usprawnienia w oparciu o istniejące biblioteki (dobór progu, kalibracja, tuning).
- **UGŁ_K01** - rozumie potrzebę korzystania z najnowszej wiedzy (literatura o wycieku danych i kryzysie powtarzalności w ML).

## Powiązanie z wykładem
- **W4** (główny): metryki klasyfikacji i regresji, kalibracja, walidacja krzyżowa, **wyciek danych i poprawny pipeline**, tuning hiperparametrów, permutation importance, raportowanie z niepewnością, kiedy sieci wygrywają z drzewami na danych tabelarycznych.
- **W3**: funkcje straty (BCE, MSE), AdamW i weight decay, Dropout, early stopping, pętla treningowa w PyTorch.
- **W1**: podział train/val/test, overfitting, bias-variance.

## Harmonogram (180 min)

| Czas | Blok | Zadania |
|---|---|---|
| 0-10 min | Wprowadzenie, 0. Przygotowanie środowiska | - |
| 10-30 min | 1. Dane tabelaryczne, podział, `Dataset` i `DataLoader` | 1.1 |
| 30-55 min | 2. MLP w PyTorch, pętla treningowa, early stopping, zapis wag | 2.1 |
| 55-80 min | 3. Baseline klasyczne i analiza wyników klasyfikacji | 3.1, 3.2 |
| 80-90 min | **Przerwa** | - |
| 90-105 min | 4. Regresja: California Housing, analiza reszt | 4.1 |
| 105-140 min | **5. Wyciek danych (data leakage) i poprawny pipeline** | 5.1, 5.2 |
| 140-155 min | 6. Walidacja krzyżowa i niepewność wyników | 6.1 |
| 155-165 min | 7. Permutation importance | 7.1 |
| 165-175 min | 8. Random search hiperparametrów | 8.1 |
| 175-180 min | Pytania kontrolne, podsumowanie | - |

## Wymagania wstępne
- Lab 1-3: tensory i autograd w PyTorch, pętla uczenia, standaryzacja, krzywe uczenia.
- Podstawy `scikit-learn` (`fit` / `predict` / `transform`).
- Wykład W4 (warto mieć pod ręką slajdy o metrykach i o wycieku danych).

## Jak pracować z notebookiem
Komórki uruchamiaj po kolei. Sekcje z teorią i kodem demonstracyjnym są kompletne - przeczytaj je uważnie, bo zadania z nich korzystają. W komórkach z `# TODO` uzupełniasz kod. Jeśli utkniesz w zadaniu 1.1 lub 2.1, poproś o pomoc - kolejne sekcje korzystają z ich wyników.

> **Konwencja w tym notebooku:** zbiór testowy traktujemy jak "sejf". Decyzje (wybór modelu, progu, hiperparametrów, epoki) podejmujemy na zbiorze walidacyjnym lub walidacją krzyżową na części deweloperskiej. Test służy wyłącznie do końcowego raportu. Z przyczyn dydaktycznych raportujemy na teście kilka modeli - w prawdziwym projekcie **wybór** spośród nich na podstawie testu byłby już wyciekiem (patrz sekcja 5.8).

## 0. Przygotowanie środowiska

**Włączenie GPU w Colab:** menu *Środowisko wykonawcze -> Zmień typ środowiska wykonawczego -> T4 GPU -> Zapisz*. Po zmianie środowisko zostanie zrestartowane.

Dla danych tabelarycznych GPU nie jest konieczne (sieci są małe), ale przyspiesza sekcje 6 i 8, w których trenujemy kilkadziesiąt sieci.

**Tryb szybki.** Zmienna `SZYBKI_TRYB` (domyślnie odczytywana ze zmiennej środowiskowej `DL_FAST`) zmniejsza liczbę epok, foldów, iteracji wyszukiwania i rozmiar zbioru California Housing. **Przełącz ją ręcznie na `True`, jeśli pracujesz bez GPU** lub chcesz najpierw sprawdzić, czy cały notebook działa. Wyniki w trybie szybkim są gorsze i bardziej zaszumione - do sprawozdania używaj trybu pełnego.

In [ ]:
# Importy - wszystkie biblioteki są domyślnie dostępne w Colab (bez pip install)
import os, time, copy, random, math, warnings
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader

from sklearn.datasets import load_breast_cancer, fetch_california_housing, make_classification
from sklearn.model_selection import (train_test_split, StratifiedKFold, KFold, GroupKFold,
                                     RepeatedStratifiedKFold, cross_val_score, cross_validate,
                                     ParameterSampler, GridSearchCV)
from sklearn.preprocessing import StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline, make_pipeline
from sklearn.feature_selection import SelectKBest, f_classif
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.ensemble import (RandomForestClassifier, HistGradientBoostingClassifier,
                              HistGradientBoostingRegressor)
from sklearn.neighbors import KNeighborsClassifier
from sklearn.dummy import DummyRegressor
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.inspection import permutation_importance
from sklearn.calibration import calibration_curve
from sklearn.metrics import (accuracy_score, balanced_accuracy_score, f1_score, precision_score,
                             recall_score, roc_auc_score, average_precision_score, brier_score_loss,
                             confusion_matrix, ConfusionMatrixDisplay, classification_report,
                             roc_curve, precision_recall_curve, mean_absolute_error,
                             mean_squared_error, r2_score)

warnings.filterwarnings('ignore', category=UserWarning)   # mniej szumu w wyjściu (np. ostrzeżenia o nazwach cech)
pd.set_option('display.precision', 4)                       # czytelniejsze tabele
plt.rcParams['figure.dpi'] = 100
print('torch', torch.__version__)

In [ ]:
# Urządzenie: GPU, jeśli jest dostępne
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('Urządzenie:', DEVICE)
if DEVICE.type == 'cuda':
    print('GPU:', torch.cuda.get_device_name(0))
else:
    print('Brak GPU - rozważ SZYBKI_TRYB = True (patrz niżej).')

# Tryb szybki: True przy pracy bez GPU (mniej epok, foldów i danych)
SZYBKI_TRYB = os.environ.get("DL_FAST", "0") == "1"
# SZYBKI_TRYB = True   # <- odkomentuj, jeśli pracujesz na CPU
if SZYBKI_TRYB:
    torch.set_num_threads(2)    # oszczędnie na słabszych maszynach

# Wszystkie "koszty obliczeniowe" w jednym miejscu
EPOKI_MAX   = 30  if SZYBKI_TRYB else 300   # górny limit epok (early stopping zwykle kończy wcześniej)
CIERPLIWOSC = 5   if SZYBKI_TRYB else 20    # patience dla early stopping
EPOKI_DEMO  = 15  if SZYBKI_TRYB else 150   # trening bez early stopping w demonstracji 2.3
EPOKI_REG   = 15  if SZYBKI_TRYB else 200   # limit epok dla regresji
N_FOLDS     = 3   if SZYBKI_TRYB else 5     # liczba foldów w CV
N_POWT      = 1   if SZYBKI_TRYB else 3     # liczba powtórzeń CV
N_ITER      = 3   if SZYBKI_TRYB else 15    # iteracje random search
N_BOOT      = 200 if SZYBKI_TRYB else 2000  # próby bootstrap
N_PERM      = 3   if SZYBKI_TRYB else 20    # powtórzenia permutacji w permutation importance
N_TREES     = 50  if SZYBKI_TRYB else 300   # liczba drzew w lasach losowych
N_CAL       = 2000 if SZYBKI_TRYB else None # rozmiar podzbioru California Housing (None = cały)
print('SZYBKI_TRYB =', SZYBKI_TRYB)

### Powtarzalność (seedy)

Sieci neuronowe są losowe na kilku poziomach: inicjalizacja wag, kolejność mini-batchy, Dropout, a w sklearn - podziały danych i losowanie drzew. Ustawiamy ziarna w `random`, `numpy` i `torch`. Na GPU część operacji cuDNN jest niedeterministyczna - flagi `deterministic=True` i `benchmark=False` to ograniczają (kosztem szybkości).

**Ważne (W4):** ustalony seed daje *powtarzalność*, a nie *rzetelność*. Wynik dla jednego seeda to jedna próbka z rozkładu wyników - dlatego w sekcji 6 będziemy raportować średnią i niepewność z wielu podziałów.

In [ ]:
SEED = 42

def ustaw_seed(seed=SEED):
    # Ustawia ziarna we wszystkich używanych generatorach liczb losowych
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True   # deterministyczne algorytmy cuDNN
    torch.backends.cudnn.benchmark = False      # bez autotuningu (który bywa niedeterministyczny)

ustaw_seed()
rng = np.random.default_rng(SEED)   # generator numpy do demonstracji w sekcji 5

## 1. Dane tabelaryczne, poprawny podział i `Dataset` / `DataLoader`

### 1.1. Zbiór Breast Cancer Wisconsin (Diagnostic)

569 pacjentek, 30 cech liczbowych opisujących jądra komórkowe z obrazu biopsji cienkoigłowej (promień, tekstura, obwód, pole, gładkość... - każda jako średnia, błąd standardowy i "najgorsza" wartość). Zadanie: rozpoznać guz **złośliwy** vs **łagodny**.

**Pułapka nr 1 - kodowanie klas.** W `sklearn` klasa `0` to *malignant* (złośliwy), a `1` to *benign* (łagodny). Metryki takie jak precision, recall, F1, PR-AUC liczone są domyślnie dla klasy `1`. Klinicznie interesuje nas wykrywanie guzów złośliwych, więc **przekodowujemy**: `y = 1` oznacza złośliwy. Zawsze sprawdzaj, która klasa jest "pozytywna" - to częste źródło błędnie zinterpretowanych wyników.

In [ ]:
bc = load_breast_cancer(as_frame=True)          # dane jako DataFrame
X_bc = bc.data                                   # 30 cech
y_bc = (1 - bc.target).astype(int)               # PRZEKODOWANIE: 1 = złośliwy (malignant)
y_bc.name = 'zlosliwy'

print('Kształt X:', X_bc.shape)
print('Liczność klas (1 = złośliwy):')
print(y_bc.value_counts().rename({0: 'łagodny (0)', 1: 'złośliwy (1)'}))
print(f'Odsetek klasy pozytywnej (prewalencja): {y_bc.mean():.3f}')
display(X_bc.iloc[:5, :8])

In [ ]:
# Skale cech różnią się o rzędy wielkości - dlatego standaryzacja jest dla sieci konieczna
opis = X_bc.describe().T[['mean', 'std', 'min', 'max']]
display(opis.sort_values('mean', ascending=False).head(6))
display(opis.sort_values('mean').head(4))

fig, ax = plt.subplots(figsize=(9, 3.5))
ax.bar(range(X_bc.shape[1]), X_bc.std().values)          # odchylenie standardowe każdej cechy
ax.set_yscale('log')                                      # skala log - inaczej małe cechy byłyby niewidoczne
ax.set_xlabel('Indeks cechy'); ax.set_ylabel('Odchylenie standardowe (skala log)')
ax.set_title('Breast Cancer: rozrzut skal cech przed standaryzacją')
plt.tight_layout(); plt.show()

### 1.2. Podział train / val / test - i dlaczego **przed** jakimkolwiek przetwarzaniem

Dzielimy dane na trzy części (W1):
- **train (60%)** - uczenie parametrów modelu (wag) **oraz parametrów przetwarzania** (średnie i odchylenia skalera, wybór cech, kategorie enkodera...),
- **val (20%)** - decyzje: wybór epoki (early stopping), progu decyzyjnego, hiperparametrów, porównanie modeli,
- **test (20%)** - jednorazowy, końcowy pomiar. Symuluje "przyszłe pacjentki", których model nigdy nie widział.

`train + val` nazywamy dalej częścią **deweloperską** (`X_dev`) - na niej w sekcjach 5-8 robimy walidację krzyżową.

**Stratyfikacja** (`stratify=y`) zachowuje proporcje klas w każdej części. Przy 569 obserwacjach bez stratyfikacji odsetek złośliwych w teście mógłby się wyraźnie różnić od populacji.

**Złota zasada pipeline'u:** *każda operacja, która czegoś "uczy się" z danych (ma metodę `fit`), jest częścią modelu.* Dlatego `StandardScaler` dopasowujemy (`fit`) **wyłącznie na train**, a na val i test tylko go stosujemy (`transform`):

$$x' = \frac{x - \hat\mu_{\text{train}}}{\hat\sigma_{\text{train}}}$$

Gdybyśmy policzyli $\hat\mu$ i $\hat\sigma$ na całym zbiorze, informacja o rozkładzie zbioru testowego "przeciekłaby" do modelu. Szczegółowo zajmiemy się tym w sekcji 5.

In [ ]:
# Krok 1: odcinamy test (20%) - od tej chwili leży "w sejfie"
X_dev, X_test, y_dev, y_test = train_test_split(
    X_bc, y_bc, test_size=0.20, stratify=y_bc, random_state=SEED)
# Krok 2: z części deweloperskiej wydzielamy walidację (0.25 * 0.8 = 20% całości)
X_train, X_val, y_train, y_val = train_test_split(
    X_dev, y_dev, test_size=0.25, stratify=y_dev, random_state=SEED)

for nazwa, yy in [('train', y_train), ('val', y_val), ('test', y_test)]:
    print(f'{nazwa:5s}: n = {len(yy):3d}, odsetek złośliwych = {yy.mean():.3f}')

# Krok 3: skaler dopasowany WYŁĄCZNIE na train
scaler = StandardScaler().fit(X_train)
X_train_s = scaler.transform(X_train)   # fit był na train -> transform
X_val_s   = scaler.transform(X_val)     # tylko transform
X_test_s  = scaler.transform(X_test)    # tylko transform

print('\nŚrednie cech po standaryzacji (train):', X_train_s.mean(0)[:4].round(3), '...')
print('Średnie cech po standaryzacji (val):  ', X_val_s.mean(0)[:4].round(3), '... (nie są dokładnie 0 - i tak ma być!)')

### 1.3. `Dataset` i `DataLoader` w PyTorch

W PyTorch dane do treningu podajemy przez dwie abstrakcje:
- **`Dataset`** - "ile jest próbek" (`__len__`) i "daj mi próbkę nr `i`" (`__getitem__`). Zwraca parę `(x_i, y_i)` jako tensory.
- **`DataLoader`** - opakowuje `Dataset`: dzieli na mini-batche (`batch_size`), tasuje kolejność co epokę (`shuffle=True` dla treningu), skleja próbki w tensory `[batch, cechy]`.

Typy danych mają znaczenie:
- cechy: `torch.float32` (domyślny typ wag w `nn.Linear`; `float64` z numpy spowodowałby błąd typu),
- etykieta dla klasyfikacji binarnej z `BCEWithLogitsLoss` oraz dla regresji z `MSELoss`: `float32` o kształcie `[n, 1]` - taki sam jak wyjście sieci z jednym neuronem. Kształt `[n]` vs `[n, 1]` to klasyczny błąd: przy regresji `MSELoss` dokona wtedy *broadcastingu* do macierzy `[n, n]` i po cichu policzy coś zupełnie innego (PyTorch wypisze jedynie ostrzeżenie).

### Zadanie 1.1 - własny `Dataset` i `DataLoader`y (ok. 8 min)

Uzupełnij klasę `TabularDataset`:
- w `__init__` zamień `X` i `y` (mogą to być DataFrame, Series lub tablice numpy) na tensory `float32`; `y` przekształć do kształtu `[n, 1]`,
- `__len__` zwraca liczbę próbek, `__getitem__(i)` zwraca krotkę `(X[i], y[i])`.

Następnie utwórz `train_loader` (batch 32, z tasowaniem), `val_loader` i `test_loader` (batch 256, bez tasowania) na danych **ustandaryzowanych**.

**Kryteria sukcesu:** komórka testowa wypisuje kształty `torch.Size([32, 30])` i `torch.Size([32, 1])`, typ `torch.float32`, a liczba batchy w `train_loader` wynosi 11.

**Wskazówki:** `torch.as_tensor(np.asarray(X), dtype=torch.float32)`, `.reshape(-1, 1)`. Dlaczego nie tasujemy walidacji? Bo kolejność nie wpływa na metrykę, a ułatwia porównywanie predykcji z etykietami.

In [ ]:
class TabularDataset(Dataset):
    def __init__(self, X, y):
        # TODO: uzupełnij - zamień X i y na tensory float32, y o kształcie [n, 1]
        self.X, self.y = None, None  # <- zastąp

    def __len__(self):
        # TODO: uzupełnij (patrz polecenie powyżej)
        return 0  # <- zastąp

    def __getitem__(self, i):
        # TODO: uzupełnij (patrz polecenie powyżej)
        return None  # <- zastąp

# TODO: uzupełnij - utwórz train_loader, val_loader, test_loader
train_loader = val_loader = test_loader = None  # <- zastąp

In [ ]:
# Komórka testowa - nie modyfikuj
xb, yb = next(iter(train_loader))
print('X batch:', xb.shape, xb.dtype)
print('y batch:', yb.shape, yb.dtype)
print('Liczba batchy w train_loader:', len(train_loader))
assert xb.shape[1] == 30 and yb.shape[1] == 1 and xb.dtype == torch.float32

## 2. MLP w PyTorch: model, pętla treningowa, early stopping, zapis wag

### 2.1. Model jako `nn.Module`

Sieć dla danych tabelarycznych to najczęściej MLP (W3): kilka warstw `Linear -> ReLU -> Dropout`. Dla klasyfikacji binarnej ostatnia warstwa ma **jeden neuron bez aktywacji** - zwraca *logit* $z$, a prawdopodobieństwo to $\hat p = \sigma(z) = 1/(1+e^{-z})$.

Dlaczego nie dodajemy `Sigmoid` na końcu? Bo używamy `nn.BCEWithLogitsLoss`, która łączy sigmoidę z entropią krzyżową w jednej, numerycznie stabilnej operacji:

$$\mathcal{L}(z, y) = -\big[y \log \sigma(z) + (1-y)\log(1-\sigma(z))\big] = \max(z,0) - zy + \log(1 + e^{-|z|}).$$

Osobne `Sigmoid` + `BCELoss` dla dużych $|z|$ daje $\log(0)$ i wartości `nan`/`inf`. To samo rozwiązanie stosuje Keras przy `from_logits=True`.

In [ ]:
class MLP(nn.Module):
    # MLP dla danych tabelarycznych: [Linear -> ReLU -> Dropout] x len(hidden) -> Linear(n_out)
    def __init__(self, n_in, hidden=(64, 32), dropout=0.2, n_out=1):
        super().__init__()
        warstwy, d = [], n_in
        for h in hidden:                                  # kolejne warstwy ukryte
            warstwy += [nn.Linear(d, h), nn.ReLU(), nn.Dropout(dropout)]
            d = h
        warstwy.append(nn.Linear(d, n_out))               # wyjście: logit (klasyfikacja) lub wartość (regresja)
        self.net = nn.Sequential(*warstwy)

    def forward(self, x):
        return self.net(x)                                # BEZ sigmoidy - zwracamy logity

ustaw_seed()
model_demo = MLP(n_in=30).to(DEVICE)
print(model_demo)
n_param = sum(p.numel() for p in model_demo.parameters())
print('Liczba parametrów:', n_param, '= (30*64+64) + (64*32+32) + (32*1+1)')

Zwróć uwagę: sieć ma ok. 4 tys. parametrów, a zbiór treningowy tylko 341 obserwacji. To typowa sytuacja dla danych tabelarycznych - stąd potrzeba regularyzacji (Dropout, weight decay) i **early stopping**.

### 2.2. Pętla treningowa - "ręczna" wersja `model.fit()`

Jedna epoka w PyTorch (porównaj z W3):
1. `model.train()` - włącza Dropout (i tryb treningowy BatchNorm),
2. dla każdego mini-batcha: `optimizer.zero_grad()` -> `loss = loss_fn(model(x), y)` -> `loss.backward()` -> `optimizer.step()`,
3. walidacja: `model.eval()` + `torch.no_grad()` - wyłączamy Dropout i śledzenie gradientów.

Średnią stratę liczymy ważąc batche ich rozmiarem (ostatni batch bywa mniejszy).

Optymalizator: **AdamW** - Adam z *odsprzężonym* weight decay (Loshchilov & Hutter, 2019). W zwykłym Adamie kara L2 dodana do gradientu jest skalowana adaptacyjnym krokiem, więc działa niespójnie; AdamW pomniejsza wagi bezpośrednio: $w \leftarrow w - \eta\lambda w$.

In [ ]:
def trenuj_epoke(model, loader, loss_fn, optimizer):
    # Jedna epoka uczenia; zwraca średnią stratę treningową
    model.train()                                          # Dropout aktywny
    suma, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)              # batch na GPU/CPU
        optimizer.zero_grad()                              # zerujemy gradienty z poprzedniego kroku
        loss = loss_fn(model(xb), yb)                      # forward + strata
        loss.backward()                                    # backprop (autograd)
        optimizer.step()                                   # aktualizacja wag
        suma += loss.item() * len(xb); n += len(xb)        # ważenie rozmiarem batcha
    return suma / n

@torch.no_grad()                                           # bez grafu obliczeń - szybciej, mniej pamięci
def ewaluuj(model, loader, loss_fn):
    # Średnia strata na zbiorze (walidacyjnym lub testowym)
    model.eval()                                           # Dropout wyłączony
    suma, n = 0.0, 0
    for xb, yb in loader:
        xb, yb = xb.to(DEVICE), yb.to(DEVICE)
        suma += loss_fn(model(xb), yb).item() * len(xb); n += len(xb)
    return suma / n

@torch.no_grad()
def przewiduj_torch(model, X, batch_size=1024):
    # Surowe wyjścia sieci (logity lub wartości regresji) jako wektor numpy
    model.eval()
    X_t = torch.as_tensor(np.asarray(X, dtype=np.float32))
    wyj = [model(X_t[i:i + batch_size].to(DEVICE)).cpu() for i in range(0, len(X_t), batch_size)]
    return torch.cat(wyj).numpy().ravel()

def proba_torch(model, X):
    # Prawdopodobieństwo klasy 1 = sigmoida z logitu
    return torch.sigmoid(torch.as_tensor(przewiduj_torch(model, X))).numpy()

### 2.3. Demonstracja: trening bez early stopping

Trenujemy przez stałą liczbę epok i obserwujemy krzywe uczenia. **Na co patrzeć:** strata treningowa maleje niemal monotonicznie; strata walidacyjna najpierw spada, potem się wypłaszcza lub rośnie - to moment, od którego model zaczyna "zapamiętywać" zbiór treningowy (overfitting, W1/W3). Krzywa walidacyjna jest poszarpana, bo walidacja ma tylko 114 obserwacji.

In [ ]:
ustaw_seed()
model_demo = MLP(n_in=30, hidden=(64, 32), dropout=0.2).to(DEVICE)
loss_fn = nn.BCEWithLogitsLoss()
opt = torch.optim.AdamW(model_demo.parameters(), lr=1e-3, weight_decay=1e-4)

hist_tr, hist_val = [], []
t0 = time.time()
for ep in range(EPOKI_DEMO):
    hist_tr.append(trenuj_epoke(model_demo, train_loader, loss_fn, opt))
    hist_val.append(ewaluuj(model_demo, val_loader, loss_fn))
print(f'Czas: {time.time() - t0:.1f} s; najniższa strata val = {min(hist_val):.4f} w epoce {int(np.argmin(hist_val)) + 1}')

plt.figure(figsize=(7, 4))
plt.plot(range(1, EPOKI_DEMO + 1), hist_tr, label='train')
plt.plot(range(1, EPOKI_DEMO + 1), hist_val, label='val')
plt.axvline(int(np.argmin(hist_val)) + 1, ls='--', c='gray', label='minimum straty val')
plt.xlabel('Epoka'); plt.ylabel('BCE'); plt.yscale('log')
plt.title('Krzywe uczenia MLP bez early stopping'); plt.legend(); plt.tight_layout(); plt.show()

### 2.4. Early stopping - teoria

Early stopping (W3) to regularyzacja przez ograniczenie czasu uczenia: przerywamy trening, gdy strata walidacyjna nie poprawiła się przez `patience` kolejnych epok, i **przywracamy wagi z najlepszej epoki**. Parametr `min_delta` określa, jaka poprawa jest "istotna" (chroni przed reagowaniem na szum).

Dwie subtelności:
1. **Kopiowanie wag.** `model.state_dict()` zwraca słownik *referencji* do tensorów modelu. Jeśli zapiszemy go bez kopiowania, "najlepsze wagi" będą się zmieniać razem z treningiem i na końcu dostaniemy wagi z ostatniej epoki. Potrzebne jest `copy.deepcopy(model.state_dict())`. To jeden z najczęstszych, "cichych" błędów w PyTorch.
2. **Early stopping zużywa zbiór walidacyjny.** Wybór epoki to wybór hiperparametru, więc strata walidacyjna najlepszej epoki jest optymistycznie obciążona. Dlatego **nigdy nie robimy early stopping na zbiorze testowym** - to częsty wyciek w publikacjach (sekcja 5.8).

### Zadanie 2.1 - klasa `EarlyStopping` (ok. 10 min)

Uzupełnij metodę `krok(val_loss, model, epoka)`, która jest wywoływana po każdej epoce i:
- jeśli `val_loss < best_loss - min_delta`: zapamiętuje nową najlepszą stratę, numer epoki i **głęboką kopię** `state_dict`, zeruje licznik,
- w przeciwnym razie zwiększa licznik epok bez poprawy,
- zwraca `True`, gdy licznik osiągnął `patience` (czyli należy przerwać trening).

Gotowa funkcja `trenuj` (komórka poniżej zadania) używa Twojej klasy - przeczytaj ją.

**Kryteria sukcesu:** komórka testowa przechodzi (sztuczna sekwencja strat), a trening w 2.5 kończy się przed limitem epok i raportuje najlepszą epokę.

**Wskazówka:** `copy.deepcopy(model.state_dict())`.

In [ ]:
class EarlyStopping:
    def __init__(self, patience=10, min_delta=0.0):
        self.patience = patience          # ile epok bez poprawy tolerujemy
        self.min_delta = min_delta        # minimalna poprawa uznawana za istotną
        self.best_loss = float('inf')     # najlepsza dotąd strata walidacyjna
        self.best_state = None            # kopia wag z najlepszej epoki
        self.best_epoch = -1
        self.licznik = 0                  # liczba epok bez poprawy

    def krok(self, val_loss, model, epoka):
        # Zwraca True, gdy należy przerwać trening
        # TODO: uzupełnij (patrz polecenie powyżej)
        return False  # <- zastąp

In [ ]:
# Komórka testowa - nie modyfikuj
_m = nn.Linear(2, 1)
_es = EarlyStopping(patience=2)
_straty = [1.0, 0.8, 0.9, 0.7, 0.75, 0.72]
_stop = []
for _ep, _l in enumerate(_straty):
    with torch.no_grad():
        _m.weight.fill_(float(_ep))          # wagi = numer epoki, żeby sprawdzić kopię
    _stop.append(_es.krok(_l, _m, _ep))
print('Decyzje stop:', _stop)
assert _stop == [False, False, False, False, False, True], 'zła logika licznika'
assert _es.best_epoch == 3 and abs(_es.best_loss - 0.7) < 1e-9
assert float(_es.best_state['weight'][0, 0]) == 3.0, 'best_state nie jest kopią (deepcopy!)'
print('EarlyStopping działa poprawnie.')

In [ ]:
def trenuj(model, train_loader, val_loader, loss_fn, lr=1e-3, weight_decay=1e-4,
           max_epoki=EPOKI_MAX, cierpliwosc=CIERPLIWOSC, verbose=True):
    # Pełny trening z early stopping; zwraca model z NAJLEPSZYMI wagami i historię
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=weight_decay)
    es = EarlyStopping(patience=cierpliwosc, min_delta=1e-4)
    hist = {'train_loss': [], 'val_loss': []}
    for ep in range(max_epoki):
        hist['train_loss'].append(trenuj_epoke(model, train_loader, loss_fn, optimizer))
        hist['val_loss'].append(ewaluuj(model, val_loader, loss_fn))
        if es.krok(hist['val_loss'][-1], model, ep):     # decyzja o zatrzymaniu
            break
    if es.best_state is not None:
        model.load_state_dict(es.best_state)            # przywracamy najlepsze wagi
    hist['best_epoch'] = es.best_epoch
    if verbose:
        print(f'Zatrzymano po {ep + 1} epokach; najlepsza epoka {es.best_epoch + 1}, '
              f'val_loss = {es.best_loss:.4f}')
    return model, hist

### 2.5. Trening z early stopping i zapis najlepszych wag

W PyTorch zapisujemy zwykle **`state_dict`** (słownik tensorów wag), a nie cały obiekt modelu - plik jest wtedy niezależny od kodu klasy i bezpieczniejszy przy wczytywaniu. Do odtworzenia potrzebujemy tej samej architektury: tworzymy nowy `MLP(...)` i wywołujemy `load_state_dict`. Do wdrożenia trzeba też zapisać **skaler** (np. `joblib.dump(scaler, ...)`) - model bez dopasowanego przetwarzania jest bezużyteczny.

In [ ]:
ustaw_seed()
mlp_bc = MLP(n_in=30, hidden=(64, 32), dropout=0.2).to(DEVICE)
t0 = time.time()
mlp_bc, hist_bc = trenuj(mlp_bc, train_loader, val_loader, nn.BCEWithLogitsLoss(), lr=1e-3, weight_decay=1e-4)
print(f'Czas treningu: {time.time() - t0:.1f} s')

plt.figure(figsize=(7, 4))
plt.plot(hist_bc['train_loss'], label='train')
plt.plot(hist_bc['val_loss'], label='val')
plt.axvline(hist_bc['best_epoch'], ls='--', c='gray', label='przywrócona epoka')
plt.xlabel('Epoka (od 0)'); plt.ylabel('BCE'); plt.yscale('log')
plt.title('MLP z early stopping'); plt.legend(); plt.tight_layout(); plt.show()

# Zapis i ponowne wczytanie najlepszych wag
torch.save(mlp_bc.state_dict(), 'mlp_breast_cancer_best.pt')
mlp_wczytany = MLP(n_in=30, hidden=(64, 32), dropout=0.2).to(DEVICE)       # ta sama architektura
mlp_wczytany.load_state_dict(torch.load('mlp_breast_cancer_best.pt', map_location=DEVICE))
roznica = np.abs(przewiduj_torch(mlp_bc, X_val_s) - przewiduj_torch(mlp_wczytany, X_val_s)).max()
print('Maksymalna różnica predykcji oryginał vs wczytany:', roznica)

## 3. Baseline klasyczne i rzetelna analiza wyników klasyfikacji

### 3.1. Po co baseline?

Sieć neuronowa na danych tabelarycznych **musi** być porównana z modelami klasycznymi (W4). Liczne prace porównawcze (np. Grinsztajn, Oyallon, Varoquaux, NeurIPS 2022, *"Why do tree-based models still outperform deep learning on tabular data?"*) pokazują, że na typowych zbiorach tabelarycznych średniej wielkości gradient boosting na drzewach (XGBoost, LightGBM, `HistGradientBoosting`) jest zwykle co najmniej tak dobry jak MLP, przy mniejszym koszcie strojenia. Bez baseline'u nie wiemy, czy 97% to dużo.

Porównujemy:
- **regresję logistyczną** (liniowa, interpretowalna, w pipeline ze skalerem),
- **las losowy** (RandomForest),
- **HistGradientBoosting** (GBDT),
- nasz **MLP**.

Modele drzewiaste nie wymagają standaryzacji (podziały są niezmiennicze względem monotonicznych przekształceń cech).

In [ ]:
# Baseline: każdy model dopasowany WYŁĄCZNIE na train; porównanie na walidacji
modele_klas = {
    'LogReg': make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000)),   # skaler w środku pipeline
    'RandomForest': RandomForestClassifier(n_estimators=N_TREES, random_state=SEED),
    'HistGB': HistGradientBoostingClassifier(random_state=SEED),
}
p_val, p_test = {}, {}
for nazwa, m in modele_klas.items():
    m.fit(X_train, y_train)                         # surowe X - pipeline LogReg sam skaluje
    p_val[nazwa] = m.predict_proba(X_val)[:, 1]     # prawdopodobieństwo klasy 1 (złośliwy)
    p_test[nazwa] = m.predict_proba(X_test)[:, 1]
p_val['MLP'] = proba_torch(mlp_bc, X_val_s)         # sieć używa danych ustandaryzowanych skalerem z train
p_test['MLP'] = proba_torch(mlp_bc, X_test_s)

def metryki_klas(y, p, prog=0.5):
    # Zestaw metryk dla klasyfikacji binarnej (p = prawdopodobieństwo klasy 1)
    yhat = (p >= prog).astype(int)
    return {'ROC-AUC': roc_auc_score(y, p), 'PR-AUC (AP)': average_precision_score(y, p),
            'Brier': brier_score_loss(y, p), 'Accuracy': accuracy_score(y, yhat),
            'Recall': recall_score(y, yhat), 'Precision': precision_score(y, yhat, zero_division=0),
            'F1': f1_score(y, yhat)}

tab_val = pd.DataFrame({n: metryki_klas(y_val, p) for n, p in p_val.items()}).T
print('Metryki na WALIDACJI (próg 0.5):')
display(tab_val.round(4))

**Jak czytać tabelę.** Na tym zbiorze wszystkie modele osiągają ROC-AUC bliskie 0,99 - różnice rzędu 0,005 przy 114 obserwacjach walidacyjnych są **w granicach szumu** (jedna pacjentka to ok. 0,9 p.p. accuracy). Nie wolno z takiej tabeli wnioskować "MLP jest najlepszy". Do porównań potrzebna jest niepewność - sekcja 6.

### 3.2. Metryki klasyfikacji - przypomnienie (W4)

Macierz pomyłek dla klasy pozytywnej "złośliwy": TP, FP, FN, TN.

- $\text{Precision} = \frac{TP}{TP+FP}$ - jaki odsetek alarmów jest trafny,
- $\text{Recall (czułość)} = \frac{TP}{TP+FN}$ - jaki odsetek złośliwych wykrywamy,
- $\text{Specificity} = \frac{TN}{TN+FP}$, $F_1 = \frac{2PR}{P+R}$,
- **ROC-AUC**: pole pod krzywą TPR(FPR) przy zmianie progu; = prawdopodobieństwo, że losowy przypadek pozytywny dostanie wyższy wynik niż losowy negatywny. Nie zależy od progu ani od prewalencji.
- **PR-AUC / Average Precision**: pole pod krzywą precision(recall). Zależy od prewalencji - model losowy ma AP równe odsetkowi pozytywnych. Przy silnym niezbalansowaniu klas jest bardziej informatywna niż ROC-AUC.
- **Brier score**: $\frac1n\sum(\hat p_i - y_i)^2$ - mierzy jednocześnie dyskryminację i kalibrację prawdopodobieństw.

Accuracy, precision, recall i F1 zależą od **progu decyzyjnego** (domyślnie 0,5), który jest decyzją, a nie właściwością modelu.

In [ ]:
# Macierz pomyłek i raport klasyfikacji dla MLP na walidacji (próg 0.5)
yhat_val_mlp = (p_val['MLP'] >= 0.5).astype(int)
print(classification_report(y_val, yhat_val_mlp, target_names=['łagodny (0)', 'złośliwy (1)'], digits=3))

fig, ax = plt.subplots(figsize=(4.5, 4))
ConfusionMatrixDisplay(confusion_matrix(y_val, yhat_val_mlp),
                       display_labels=['łagodny', 'złośliwy']).plot(ax=ax, cmap='Blues', colorbar=False)
ax.set_title('MLP - macierz pomyłek (walidacja, próg 0.5)')
ax.set_xlabel('Predykcja'); ax.set_ylabel('Prawda')
plt.tight_layout(); plt.show()

In [ ]:
# Krzywe ROC i PR dla wszystkich modeli (walidacja)
fig, axes = plt.subplots(1, 2, figsize=(12, 4.5))
for nazwa, p in p_val.items():
    fpr, tpr, _ = roc_curve(y_val, p)
    axes[0].plot(fpr, tpr, label=f'{nazwa} (AUC={roc_auc_score(y_val, p):.3f})')
    prec, rec, _ = precision_recall_curve(y_val, p)
    axes[1].plot(rec, prec, label=f'{nazwa} (AP={average_precision_score(y_val, p):.3f})')
axes[0].plot([0, 1], [0, 1], 'k--', lw=1, label='losowy')
axes[0].set_xlabel('FPR (1 - specyficzność)'); axes[0].set_ylabel('TPR (czułość)'); axes[0].set_title('Krzywe ROC (walidacja)')
axes[1].axhline(y_val.mean(), color='k', ls='--', lw=1, label=f'losowy (AP = prewalencja = {y_val.mean():.2f})')
axes[1].set_xlabel('Recall'); axes[1].set_ylabel('Precision'); axes[1].set_title('Krzywe precision-recall (walidacja)')
for a in axes: a.legend(loc='lower right' if a is axes[0] else 'lower left', fontsize=8); a.grid(alpha=.3)
plt.tight_layout(); plt.show()

### 3.3. Dobór progu decyzyjnego

Próg 0,5 jest optymalny tylko przy równych kosztach błędów i dobrze skalibrowanych prawdopodobieństwach. W diagnostyce onkologicznej przeoczenie guza złośliwego (FN) jest dużo kosztowniejsze niż dodatkowa biopsja (FP). Typowa specyfikacja: "**czułość co najmniej 98%**, przy możliwie wysokiej precyzji".

**Kluczowe:** próg to hiperparametr. Wybieramy go na **walidacji**, a jakość oceniamy na **teście**. Wybór progu na teście da optymistyczny wynik (np. "czułość 100%"), który nie powtórzy się na nowych danych. Przy małej walidacji próg jest niepewny - realna czułość na teście może spaść poniżej celu; warto to skomentować w raporcie.

### Zadanie 3.1 - próg o zadanej czułości (ok. 8 min)

1. Na **walidacji** policz `precision_recall_curve(y_val, p_val['MLP'])`. Uwaga: tablice `precision` i `recall` mają o jeden element więcej niż `thresholds` - odrzuć ostatni element.
2. Spośród progów, dla których `recall >= 0.98`, wybierz ten o **najwyższej precyzji** (`prog_opt`). Jeśli żaden nie spełnia warunku, użyj 0,5.
3. Narysuj precision i recall w funkcji progu (walidacja), zaznacz `prog_opt` i próg 0,5.
4. Zastosuj `prog_opt` na **teście**: wypisz recall, precision, specyficzność i macierz pomyłek; porównaj z progiem 0,5.

**Kryteria sukcesu:** wykres z legendą i tabelka dla obu progów na teście. Skomentuj, czy czułość na teście osiągnęła 98%, a jeśli nie - dlaczego. Zwróć uwagę, że wybrany próg leży dokładnie "na krawędzi" warunku na walidacji (ze 114 obserwacji, w tym ok. 43 złośliwych) - jak można by zwiększyć margines bezpieczeństwa?

In [ ]:
# TODO: uzupełnij (patrz polecenie powyżej)

### 3.4. Kalibracja prawdopodobieństw

Model jest **skalibrowany**, jeśli wśród przypadków, którym przypisał $\hat p \approx 0{,}8$, ok. 80% to faktycznie pozytywy: $P(Y=1 \mid \hat p = q) = q$. Dobra dyskryminacja (wysokie AUC) **nie** gwarantuje kalibracji - AUC zależy tylko od kolejności predykcji.

**Krzywa kalibracji (reliability diagram):** dzielimy predykcje na przedziały, w każdym rysujemy średnie $\hat p$ (oś X) vs odsetek pozytywów (oś Y). Idealny model leży na przekątnej. Typowe wzorce: las losowy - prawdopodobieństwa "ściśnięte" do środka; sieci trenowane długo bez regularyzacji - zbyt pewne siebie (Guo i in., ICML 2017).

**Rekalibracja:** skalowanie Platta (regresja logistyczna na logicie), *temperature scaling*, regresja izotoniczna. Kalibrator to kolejny element "uczony z danych" - dopasowujemy go na **walidacji**, a oceniamy na **teście**. Dopasowanie na danych treningowych modelu nic nie da (model jest tam zbyt pewny), a na teście - to wyciek.

**Zastrzeżenie:** przy 114 obserwacjach krzywe kalibracji są bardzo zaszumione. Używamy 5 przedziałów kwantylowych i traktujemy wynik jakościowo.

### Zadanie 3.2 - kalibracja i skalowanie Platta (ok. 8 min)

1. Narysuj krzywe kalibracji na **teście** dla MLP, LogReg i RandomForest (`calibration_curve(y, p, n_bins=5, strategy='quantile')`), z przekątną i legendą zawierającą Brier score.
2. Zaimplementuj skalowanie Platta dla MLP: dopasuj `LogisticRegression` na **logitach walidacyjnych** MLP (`przewiduj_torch(mlp_bc, X_val_s).reshape(-1, 1)`) i `y_val`; zastosuj na logitach testowych.
3. Porównaj Brier score MLP na teście przed i po kalibracji oraz dodaj krzywą po kalibracji do wykresu.

**Kryteria sukcesu:** wykres z 4 krzywymi; tabelka Brier przed/po. Skomentuj: czy kalibracja zmieniła ROC-AUC? (Nie powinna - Platt jest monotoniczny.)

In [ ]:
# TODO: uzupełnij (patrz polecenie powyżej)

## 4. Regresja: California Housing i analiza reszt

Zbiór California Housing (spis 1990): 20 640 dzielnic, 8 cech (mediana dochodu, wiek budynków, liczba pokoi, populacja, położenie...), cel: mediana wartości domu w setkach tysięcy USD.

### Metryki regresji (W4)
- $\text{MAE} = \frac1n\sum|y_i - \hat y_i|$ - w jednostkach celu, odporna na wartości odstające,
- $\text{RMSE} = \sqrt{\frac1n\sum(y_i - \hat y_i)^2}$ - mocniej karze duże błędy; RMSE $\ge$ MAE, duża różnica sygnalizuje ciężkie ogony błędów,
- $R^2 = 1 - \frac{\sum(y_i-\hat y_i)^2}{\sum(y_i-\bar y)^2}$ - jaki ułamek wariancji wyjaśnia model w porównaniu z przewidywaniem średniej; może być ujemne na teście.

### Analiza reszt
Reszta $r_i = y_i - \hat y_i$. Na wykresie reszt vs $\hat y$ szukamy: (1) struktury (krzywizna -> model nie uchwycił nieliniowości), (2) zmiennej wariancji (heteroskedastyczność), (3) artefaktów danych. **W tym zbiorze cel jest ucięty na wartości 5,0** (wszystkie droższe domy zapisano jako 5,0) - na wykresie $y$ vs $\hat y$ zobaczysz poziomą linię, a reszty dla tych dzielnic są systematycznie dodatnie. To przykład, jak analiza reszt odkrywa właściwości procesu zbierania danych.

Baseline: `DummyRegressor` (średnia z train - punkt odniesienia dla $R^2 = 0$), `Ridge` (liniowa) i `HistGradientBoostingRegressor`.

In [ ]:
cal = fetch_california_housing(as_frame=True)
X_cal, y_cal = cal.data, cal.target
if N_CAL is not None and len(X_cal) > N_CAL:              # tryb szybki: losowy podzbiór
    idx = np.random.default_rng(SEED).choice(len(X_cal), N_CAL, replace=False)
    X_cal, y_cal = X_cal.iloc[idx], y_cal.iloc[idx]
print('Kształt:', X_cal.shape, '| cel min/max:', round(float(y_cal.min()), 3), round(float(y_cal.max()), 3))
print('Odsetek obserwacji z celem >= 5.0 (ucięcie):', round(float((y_cal >= 5.0).mean()), 4))

# Podział 60/20/20 - analogicznie jak wyżej
Xc_dev, Xc_test, yc_dev, yc_test = train_test_split(X_cal, y_cal, test_size=0.2, random_state=SEED)
Xc_train, Xc_val, yc_train, yc_val = train_test_split(Xc_dev, yc_dev, test_size=0.25, random_state=SEED)
sc_cal = StandardScaler().fit(Xc_train)                    # skaler tylko na train
Xc_train_s, Xc_val_s, Xc_test_s = (sc_cal.transform(a) for a in (Xc_train, Xc_val, Xc_test))

plt.figure(figsize=(6, 3.5))
plt.hist(yc_train, bins=50, color='steelblue')
plt.xlabel('Mediana wartości domu [100 tys. USD]'); plt.ylabel('Liczba dzielnic')
plt.title('Rozkład celu (train) - zwróć uwagę na pik przy 5.0'); plt.tight_layout(); plt.show()

In [ ]:
# Baseline'y regresji (dopasowanie na train, ocena na teście w zadaniu 4.1)
modele_reg = {'Dummy (średnia)': DummyRegressor(),
              'Ridge': make_pipeline(StandardScaler(), Ridge(alpha=1.0)),
              'HistGB': HistGradientBoostingRegressor(random_state=SEED)}
pred_reg_test = {}
for nazwa, m in modele_reg.items():
    m.fit(Xc_train, yc_train)
    pred_reg_test[nazwa] = m.predict(Xc_test)

def metryki_reg(y, yhat):
    return {'MAE': mean_absolute_error(y, yhat), 'RMSE': math.sqrt(mean_squared_error(y, yhat)),
            'R2': r2_score(y, yhat)}

display(pd.DataFrame({n: metryki_reg(yc_test, p) for n, p in pred_reg_test.items()}).T.round(4))

### Zadanie 4.1 - MLP regresyjny i analiza reszt (ok. 12 min)

1. Utwórz `DataLoader`y (`TabularDataset` z zadania 1.1, batch 64 dla train) na `Xc_*_s` i `yc_*`.
2. Zbuduj `MLP(n_in=8, hidden=(128, 64), dropout=0.1)` i wytrenuj funkcją `trenuj` ze stratą `nn.MSELoss()`, `lr=1e-3`, `max_epoki=EPOKI_REG`.
3. Policz predykcje na teście (`przewiduj_torch`), dodaj je do `pred_reg_test['MLP']` i wypisz tabelę MAE/RMSE/R2 dla wszystkich modeli.
4. Narysuj trzy wykresy dla MLP: (a) $y$ vs $\hat y$ z przekątną, (b) reszty vs $\hat y$ z linią 0, (c) histogram reszt.

**Kryteria sukcesu:** MLP wyraźnie lepszy od Ridge i Dummy (w pełnym trybie $R^2 \approx 0{,}78$-$0{,}81$). Porównaj z HistGB i skomentuj ucięcie przy 5,0.

**Wskazówka:** opcjonalnie możesz standaryzować także cel - wtedy pamiętaj o odwróceniu transformacji przed liczeniem metryk.

In [ ]:
# TODO: uzupełnij (patrz polecenie powyżej)

## 5. Wyciek danych (data leakage) i poprawny pipeline

### 5.1. Definicja

**Wyciek danych** zachodzi, gdy model lub procedura jego budowy korzysta z informacji, która **nie byłaby dostępna w momencie rzeczywistej predykcji** - najczęściej z informacji o zbiorze, na którym potem mierzymy jakość. Skutek: estymata błędu uogólnienia jest **optymistycznie obciążona**, czasem drastycznie. Model wygląda świetnie w artykule i zawodzi po wdrożeniu.

Formalnie: chcemy oszacować $\text{Err} = \mathbb{E}_{(x,y)\sim \mathcal{D}}\,[\ell(f_{\mathcal{T}}(x), y)]$, gdzie $f_{\mathcal{T}}$ to **cała procedura** uczenia (przetwarzanie + selekcja cech + dobór hiperparametrów + trening) zastosowana do zbioru uczącego $\mathcal{T}$. Estymata na zbiorze testowym $\mathcal{S}$ jest nieobciążona tylko wtedy, gdy $\mathcal{S}$ jest **niezależny** od $f_{\mathcal{T}}$, czyli gdy żaden krok procedury nie "widział" $\mathcal{S}$ - ani jego etykiet, ani nawet rozkładu cech - i gdy obserwacje w $\mathcal{S}$ są niezależne od obserwacji w $\mathcal{T}$.

Najważniejsza konsekwencja:

> **Walidujemy procedurę, a nie model.** Wszystko, co jest dopasowywane do danych, musi znaleźć się *wewnątrz* pętli walidacji i być dopasowywane od nowa w każdym foldzie, wyłącznie na jego części treningowej.

Problem jest poważny i powszechny: przegląd Kapoor i Narayanan (*Patterns*, 2023) zidentyfikował wyciek w kilkuset publikacjach z kilkunastu dziedzin stosujących ML, często prowadzący do zawyżonych lub nieodtwarzalnych wyników. Klasyczne prace: Ambroise i McLachlan (*PNAS*, 2002) - błąd selekcji cech przed walidacją krzyżową w danych mikromacierzowych; Kaufman i in. (*ACM TKDD*, 2012) - formalizacja wycieku; Cawley i Talbot (*JMLR*, 2010) - nadmierne dopasowanie przy selekcji modelu; Varma i Simon (*BMC Bioinformatics*, 2006) - obciążenie CV przy strojeniu hiperparametrów.

### 5.2. Taksonomia wycieków

| # | Typ wycieku | Przykład | Siła efektu | Jak wykryć | Jak zapobiec |
|---|---|---|---|---|---|
| 1 | Przetwarzanie dopasowane na całości (bez użycia $y$) | `StandardScaler`, imputacja średnią, PCA na całym zbiorze przed podziałem | zwykle mała, rośnie przy małym $n$ i przesunięciu rozkładu | przegląd kodu: `fit` przed `split` | `Pipeline`, `fit` tylko na train |
| 2 | Przetwarzanie z użyciem etykiet na całości | selekcja cech (test t, korelacja z $y$, LASSO), target encoding | **duża**, szczególnie przy $p \gg n$ | wynik na losowych etykietach >> szansa | selekcja wewnątrz foldu (`Pipeline`) |
| 3 | Oversampling / augmentacja przed podziałem | SMOTE lub duplikowanie klasy mniejszościowej, augmentacja obrazów | duża | te same lub prawie identyczne wiersze w train i test | resampling tylko na części treningowej foldu |
| 4 | Cechy z przyszłości / pośrednik celu (*target leakage*) | "podano chemioterapię", "data wypisu", identyfikator ośrodka korelujący z diagnozą | **bardzo duża**, często AUC ~ 1 | jedna cecha dominuje w ważności; zbyt dobry wynik | analiza, *kiedy* cecha jest znana; konsultacja z ekspertem domenowym |
| 5 | Zależne obserwacje (grupy) i duplikaty | wiele próbek od tej samej pacjentki, zwierzęcia, kłody, sesji EEG; zduplikowane rekordy | duża | model "rozpoznaje osobę"; spadek przy podziale po grupach | `GroupKFold`, deduplikacja przed podziałem |
| 6 | Wyciek czasowy | losowy podział szeregu czasowego, cechy liczone z oknem obejmującym przyszłość | duża | spadek przy podziale "przeszłość -> przyszłość" | `TimeSeriesSplit`, podział po dacie |
| 7 | Wyciek przez selekcję modelu | strojenie hiperparametrów lub early stopping na teście, wielokrotne "podglądanie" testu, raport najlepszego wyniku z wielu prób | umiarkowana do dużej (rośnie z liczbą prób) | wynik CV wybranej konfiguracji > wynik zagnieżdżonej CV | zagnieżdżona CV; test używany raz; rejestrowanie wszystkich prób |
| 8 | Niereprezentatywny test | test z innego rozkładu niż populacja docelowa, test "oczyszczony" ręcznie | dowolna | analiza pochodzenia danych | walidacja zewnętrzna (inny ośrodek, inny okres) |

W kolejnych podsekcjach **zademonstrujemy eksperymentalnie** typy 1, 2, 3, 4 i 5, a typ 7 - w sekcji 8 i w zadaniu dodatkowym. Wszystkie eksperymenty robimy na części deweloperskiej lub na danych syntetycznych - test Breast Cancer pozostaje nietknięty.

### 5.3. Typ 1: skaler dopasowany na całym zbiorze

Porównujemy dwie procedury walidacji krzyżowej regresji logistycznej:
- **z wyciekiem:** `StandardScaler().fit(X_dev)` na całej części deweloperskiej, potem CV,
- **poprawnie:** `Pipeline(StandardScaler, LogisticRegression)` - skaler dopasowywany od nowa w każdym foldzie.

Spodziewaj się **małej** różnicy. Standaryzacja nie używa etykiet, a średnie i odchylenia z 455 obserwacji są stabilne. Uczciwie trzeba powiedzieć: na dużym, jednorodnym zbiorze ten wyciek jest zwykle liczbowo pomijalny. Mimo to:
- jest to **błąd metodologiczny** - procedura nie symuluje wdrożenia (we wdrożeniu nie znamy statystyk przyszłych danych),
- efekt rośnie przy małych zbiorach, przy przetwarzaniu bardziej "elastycznym" (PCA na wielu wymiarach, imputacja kNN/iteracyjna, normalizacja per-zbiór) i przy przesunięciu rozkładu między train a test (np. inny ośrodek, inny skaner),
- recenzent nie jest w stanie ocenić, czy efekt był mały - więc kod musi być poprawny z definicji.

In [ ]:
cv5 = RepeatedStratifiedKFold(n_splits=5, n_repeats=N_POWT, random_state=SEED)   # te same foldy dla obu procedur

# Z WYCIEKIEM: skaler "widzi" wszystkie obserwacje, także przyszłe foldy testowe
X_dev_leak = StandardScaler().fit_transform(X_dev)
auc_leak = cross_val_score(LogisticRegression(max_iter=5000), X_dev_leak, y_dev, cv=cv5, scoring='roc_auc')

# POPRAWNIE: skaler jest krokiem pipeline'u -> dopasowywany w każdym foldzie tylko na jego train
pipe_ok = make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))
auc_ok = cross_val_score(pipe_ok, X_dev, y_dev, cv=cv5, scoring='roc_auc')

print(f'Skaler na całości: ROC-AUC = {auc_leak.mean():.4f} +/- {auc_leak.std():.4f}')
print(f'Skaler w pipeline: ROC-AUC = {auc_ok.mean():.4f} +/- {auc_ok.std():.4f}')
print(f'Średnia różnica (parami, te same foldy): {np.mean(auc_leak - auc_ok):+.5f}')

### 5.4. Typ 2: selekcja cech przed walidacją - eksperyment na czystym szumie

To najbardziej pouczający eksperyment (Ambroise i McLachlan, 2002; Hastie, Tibshirani, Friedman, *The Elements of Statistical Learning*, rozdz. 7.10.2 "The Wrong and Right Way to Do Cross-validation").

Generujemy **czysty szum**: $n = 100$ obserwacji, $p$ = kilka tysięcy cech $\sim \mathcal{N}(0,1)$, etykiety losowane rzutem monetą, **niezależnie od cech**. Żaden model nie może tu przewidywać lepiej niż 50%. Prawdziwa dokładność każdej procedury wynosi 0,5.

Procedura "z wyciekiem" (spotykana w publikacjach z danymi omicznymi, obrazowymi, EEG):
1. wybierz $k = 20$ cech najbardziej skorelowanych z $y$ (test F) **na całym zbiorze**,
2. zrób 5-krotną CV klasyfikatora na wybranych cechach.

Wśród tysięcy losowych cech zawsze znajdzie się kilkadziesiąt, które *przypadkiem* korelują z $y$ w tej konkretnej próbie - **także w obserwacjach, które później trafią do foldów testowych**. Selekcja "zapamiętała" etykiety testowe.

In [ ]:
n_szum, p_szum, k_szum = 100, (2000 if SZYBKI_TRYB else 5000), 20
X_szum = rng.normal(size=(n_szum, p_szum))              # cechy: czysty szum
y_szum = rng.permutation(np.repeat([0, 1], n_szum // 2))  # etykiety niezależne od cech (50/50)
cv_szum = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)

# Z WYCIEKIEM: selekcja cech na CAŁYM zbiorze (używa etykiet foldów testowych!)
selektor = SelectKBest(f_classif, k=k_szum).fit(X_szum, y_szum)
X_szum_sel = selektor.transform(X_szum)
acc_szum_leak = cross_val_score(LogisticRegression(max_iter=2000), X_szum_sel, y_szum, cv=cv_szum)
print(f'Selekcja przed CV:  accuracy = {acc_szum_leak.mean():.3f}  (prawdziwa wartość: 0.5!)')

### Zadanie 5.1 - poprawna selekcja cech i wpływ $k$ (ok. 10 min)

1. Zbuduj `Pipeline` z krokami `('select', SelectKBest(f_classif, k=20))` i `('clf', LogisticRegression(max_iter=2000))` i oceń go `cross_val_score` z tym samym `cv_szum` na **pełnym** `X_szum`. Wynik zapisz w `acc_szum_ok`.
2. Powtórz obie procedury (z wyciekiem i poprawną) dla `k` w `[5, 20, 100, 500]` i narysuj wykres: oś X - $k$ (skala log), oś Y - średnia accuracy, dwie linie oraz pozioma linia na poziomie 0,5 (szansa).
3. Zinterpretuj: dla jakiego $k$ wyciek jest najsilniejszy i dlaczego?

**Kryteria sukcesu:** procedura poprawna oscyluje wokół 0,5 (w granicach ok. +/- 0,1), procedura z wyciekiem jest wyraźnie wyżej.

**Wskazówka:** w procedurze z wyciekiem `SelectKBest` dopasowujesz na całym `X_szum` dla każdego `k` osobno.

In [ ]:
# TODO: uzupełnij (patrz polecenie powyżej)
acc_szum_ok = None  # <- zastąp

### 5.5. Typ 3: oversampling przed podziałem

Przy niezbalansowanych klasach popularne jest powielanie (lub syntetyczne generowanie, SMOTE) przykładów klasy mniejszościowej. Jeśli zrobimy to **przed** podziałem, kopie tego samego pacjenta trafią jednocześnie do części treningowej i testowej foldu. Model, który zapamiętuje przykłady (las losowy, kNN, duża sieć), "rozpozna" je w teście. SMOTE daje ten sam problem w łagodniejszej postaci - syntetyczne punkty są interpolacjami między sąsiadami, więc informacja o punktach testowych trafia do treningu.

Poprawnie: resampling jest **częścią procedury uczenia** - wykonujemy go tylko na części treningowej każdego foldu, a fold testowy pozostaje w oryginalnym, niezbalansowanym rozkładzie (takim jak we wdrożeniu). W bibliotece `imbalanced-learn` służy do tego `imblearn.pipeline.Pipeline`; tutaj zrobimy to ręcznie.

In [ ]:
# Dane niezbalansowane (ok. 10% klasy 1) o umiarkowanie trudnym sygnale
X_imb, y_imb = make_classification(n_samples=600, n_features=10, n_informative=3, n_redundant=2,
                                   weights=[0.9, 0.1], class_sep=0.6, flip_y=0.05, random_state=SEED)

def oversample(X, y, gen):
    # Losowe powielanie klasy 1 do liczności klasy 0 (z powtórzeniami)
    i_min, i_maj = np.where(y == 1)[0], np.where(y == 0)[0]
    extra = gen.choice(i_min, size=len(i_maj) - len(i_min), replace=True)
    idx = np.concatenate([i_maj, i_min, extra])
    return X[idx], y[idx]

cv_imb = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
rf = RandomForestClassifier(n_estimators=N_TREES, random_state=SEED)

# Z WYCIEKIEM: oversampling całego zbioru, potem CV
X_os, y_os = oversample(X_imb, y_imb, np.random.default_rng(SEED))
f1_os_leak = cross_val_score(rf, X_os, y_os, cv=cv_imb, scoring='f1')

# POPRAWNIE: oversampling tylko w części treningowej foldu; fold testowy w oryginalnym rozkładzie
f1_os_ok = []
gen = np.random.default_rng(SEED)
for tr, te in cv_imb.split(X_imb, y_imb):
    Xtr, ytr = oversample(X_imb[tr], y_imb[tr], gen)
    rf.fit(Xtr, ytr)
    f1_os_ok.append(f1_score(y_imb[te], rf.predict(X_imb[te])))

print(f'Prewalencja klasy 1: {y_imb.mean():.3f}')
print(f'Oversampling przed CV: F1 = {f1_os_leak.mean():.3f}')
print(f'Oversampling w foldzie: F1 = {np.mean(f1_os_ok):.3f}')

Różnica jest dramatyczna: model z wyciekiem "wykrywa" klasę mniejszościową niemal bezbłędnie, bo w każdym foldzie testowym jest mnóstwo kopii przykładów, które widział w treningu. Dodatkowo fold testowy ma rozkład 50/50 zamiast 90/10, więc nawet precision jest liczona w nierealistycznych warunkach.

### 5.6. Typ 4: cecha-pośrednik celu (target leakage)

Do danych o umiarkowanym sygnale dodajemy cechę, która w rzeczywistości jest **skutkiem** etykiety, np. "skierowanie na chemioterapię" w modelu diagnozującym nowotwór, "liczba dni hospitalizacji" w modelu przewidującym powikłania przy przyjęciu, "wynik badania histopatologicznego" przy predykcji z obrazu USG. Taka cecha jest w bazie danych, ale **nie istnieje w momencie, w którym model miałby być użyty**.

Takiego wycieku nie usunie żaden `Pipeline` - to błąd w **definicji problemu**. Wykrywamy go: (1) pytając dla każdej cechy "kiedy i przez kogo jest ona rejestrowana?", (2) traktując podejrzliwie wyniki "zbyt dobre, by były prawdziwe", (3) analizując ważność cech - jedna cecha dominująca nad wszystkimi to sygnał ostrzegawczy (zapowiedź sekcji 7).

In [ ]:
# Umiarkowany sygnał: AUC ok. 0.85
X_w, y_w = make_classification(n_samples=1000, n_features=8, n_informative=3, n_redundant=0,
                               class_sep=0.3, flip_y=0.15, random_state=SEED)
cecha_po_diagnozie = y_w + rng.normal(0, 0.4, size=len(y_w))     # skutek etykiety + szum
X_wl = np.column_stack([X_w, cecha_po_diagnozie])
nazwy_wl = [f'x{i}' for i in range(8)] + ['skierowanie_po_diagnozie']

Xa, Xb, ya, yb = train_test_split(X_wl, y_w, test_size=0.3, stratify=y_w, random_state=SEED)
hgb_bez = HistGradientBoostingClassifier(random_state=SEED).fit(Xa[:, :8], ya)
hgb_z = HistGradientBoostingClassifier(random_state=SEED).fit(Xa, ya)
print(f'ROC-AUC bez cechy-pośrednika: {roc_auc_score(yb, hgb_bez.predict_proba(Xb[:, :8])[:, 1]):.3f}')
print(f'ROC-AUC Z cechą-pośrednikiem: {roc_auc_score(yb, hgb_z.predict_proba(Xb)[:, 1]):.3f}  <- "zbyt dobre, by było prawdziwe"')

# Diagnostyka: permutation importance na zbiorze wstrzymanym
pi_wl = permutation_importance(hgb_z, Xb, yb, scoring='roc_auc', n_repeats=N_PERM, random_state=SEED)
kol = ['crimson' if n == 'skierowanie_po_diagnozie' else 'steelblue' for n in nazwy_wl]
plt.figure(figsize=(7, 3.5))
plt.barh(nazwy_wl, pi_wl.importances_mean, xerr=pi_wl.importances_std, color=kol)
plt.xlabel('Spadek ROC-AUC po permutacji cechy'); plt.title('Jedna cecha dominuje - sygnał ostrzegawczy wycieku')
plt.tight_layout(); plt.show()

### 5.7. Typ 5: zależne obserwacje - wiele próbek od tej samej osoby

W danych biomedycznych często mamy kilka obserwacji na jednostkę: kilka wizyt tej samej pacjentki, kilka epok EEG z jednej sesji, kilka zdjęć tej samej zmiany skórnej, kilka sekcji tej samej kłody drewna. Obserwacje z jednej jednostki są do siebie podobne ("odcisk palca" osoby). Przy losowym podziale wierszy próbki tej samej osoby trafiają do treningu i testu - model może nauczyć się **rozpoznawać osobę**, a nie zjawisko, i nadal uzyskać świetny wynik.

Właściwe pytanie brzmi: *na jakiej jednostce model będzie stosowany?* Jeśli na **nowych pacjentkach** - podział musi być **po pacjentkach** (`GroupKFold`, `StratifiedGroupKFold`, `GroupShuffleSplit`). Wszystkie próbki danej osoby lądują wtedy po jednej stronie podziału.

Symulacja poniżej jest skrajna celowo: etykieta jest przypisana do pacjentki losowo i **nie zależy od cech** (prawdziwa dokładność = 0,5), ale każda pacjentka ma własny "odcisk" w cechach.

In [ ]:
n_pac, m_prob, d_g = (40 if SZYBKI_TRYB else 80), 5, 20
odcisk = rng.normal(size=(n_pac, d_g))                               # "odcisk palca" każdej pacjentki
X_g = np.repeat(odcisk, m_prob, axis=0) + rng.normal(0, 0.3, size=(n_pac * m_prob, d_g))  # 5 podobnych próbek na osobę
y_pac = rng.permutation(np.repeat([0, 1], n_pac // 2))               # etykieta losowa, niezależna od cech
y_g = np.repeat(y_pac, m_prob)
grupy = np.repeat(np.arange(n_pac), m_prob)                          # identyfikator pacjentki

modele_g = {'RandomForest': RandomForestClassifier(n_estimators=N_TREES, random_state=SEED),
            'kNN (k=1)': KNeighborsClassifier(n_neighbors=1)}
kf = KFold(n_splits=5, shuffle=True, random_state=SEED)              # losowy podział WIERSZY
acc_kf = {n: cross_val_score(m, X_g, y_g, cv=kf).mean() for n, m in modele_g.items()}
for n, a in acc_kf.items():
    print(f'{n:13s} KFold po wierszach: accuracy = {a:.3f}   (prawdziwa wartość: 0.5)')

### Zadanie 5.2 - walidacja z podziałem po grupach (ok. 8 min)

1. Oceń oba modele z `modele_g` przy użyciu `GroupKFold(n_splits=5)` (pamiętaj o argumencie `groups=grupy` w `cross_val_score`). Wyniki zapisz w słowniku `acc_gkf`.
2. Sprawdź asercją, że w żadnym foldzie `GroupKFold` żadna pacjentka nie występuje jednocześnie w train i test (iteruj po `GroupKFold(5).split(X_g, y_g, grupy)` i porównaj zbiory `set(grupy[tr])`, `set(grupy[te])`). Zrób to samo dla `kf` i wypisz, w ilu foldach `KFold` takie przecięcie jest niepuste.
3. Narysuj wykres słupkowy: dla każdego modelu dwa słupki (KFold vs GroupKFold) i linia szansy 0,5.

**Kryteria sukcesu:** KFold daje wyniki bliskie 1, GroupKFold - okolice 0,5. Jedno zdanie wniosku: co tak naprawdę mierzyła walidacja po wierszach?

In [ ]:
# TODO: uzupełnij (patrz polecenie powyżej)

### 5.8. Typ 7: wyciek przez selekcję modelu i "podglądanie" testu

Ten wyciek nie wymaga żadnego błędu w kodzie przetwarzania. Wystarczy, że:
- stroimy hiperparametry (lub epokę early stopping, próg, architekturę) patrząc na wynik testowy,
- uruchamiamy wiele wariantów i raportujemy najlepszy (*"garden of forking paths"*),
- wracamy do testu "tylko jeszcze raz" po każdej zmianie.

Każda decyzja podjęta na podstawie wyniku jest dopasowaniem do szumu w tym zbiorze. Przy $M$ porównywanych konfiguracjach o identycznej prawdziwej jakości, najlepsza z nich ma wynik zawyżony o wartość rosnącą mniej więcej jak $\sigma\sqrt{2\ln M}$ (maksimum $M$ zmiennych gaussowskich) - tzw. *winner's curse*. Remedium:
1. decyzje na walidacji / CV na części deweloperskiej, test raz na końcu,
2. **zagnieżdżona walidacja krzyżowa** (pętla wewnętrzna - strojenie, pętla zewnętrzna - ocena całej procedury strojenia) - zadanie dodatkowe,
3. raportowanie wszystkich prób, a nie tylko najlepszej.

### 5.9. Poprawny pipeline - podsumowanie

```
             DANE (po deduplikacji, z identyfikatorem grupy, z cechami dostępnymi w chwili predykcji)
                 |
     podział (stratyfikowany / po grupach / po czasie)  -> TEST (sejf, użyty raz)
                 |
           część deweloperska
                 |
   +------------ pętla CV (każdy fold osobno) ----------------------------+
   |  fit na train foldu:  imputacja -> skalowanie -> selekcja cech ->    |
   |                       resampling -> model (+ early stopping na       |
   |                       wewnętrznym podziale train foldu)              |
   |  transform + predict na teście foldu -> metryka                      |
   +----------------------------------------------------------------------+
                 |
   wybór hiperparametrów (lub zagnieżdżona CV) -> trening na całej części deweloperskiej
                 |
           jednorazowa ocena na TEŚCIE + przedział niepewności
```

W `scikit-learn` mechanizmem gwarantującym poprawność jest **`Pipeline`**: `cross_val_score`, `GridSearchCV` i `permutation_importance` klonują cały pipeline i wywołują `fit` od zera na każdym podzbiorze uczącym. Aby skorzystać z tego mechanizmu dla sieci w PyTorch, opakujemy ją w estymator zgodny ze sklearn (metody `fit`, `predict_proba`, `predict`, parametry w `__init__`). Early stopping wewnątrz `fit` korzysta z **wewnętrznego** podziału danych przekazanych do `fit` - nigdy z foldu testowego.

In [ ]:
# Przykład kompletnego pipeline'u sklearn: wszystkie kroki "uczone z danych" są wewnątrz
pipe_pelny = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),    # (tu brak braków danych, ale krok jest bezpieczny)
    ('scaler', StandardScaler()),
    ('select', SelectKBest(f_classif, k=15)),
    ('clf', LogisticRegression(max_iter=5000)),
])
wyn = cross_validate(pipe_pelny, X_dev, y_dev, cv=StratifiedKFold(5, shuffle=True, random_state=SEED),
                     scoring=['roc_auc', 'balanced_accuracy'], return_estimator=True)
print('ROC-AUC per fold:', wyn['test_roc_auc'].round(4))
# Każdy fold wybrał (nieco) inne cechy - to normalne i to właśnie musi zostać uwzględnione w ocenie
for i, est in enumerate(wyn['estimator'][:3]):
    wybrane = np.array(X_dev.columns)[est.named_steps['select'].get_support()]
    print(f'fold {i}: {sorted(wybrane)[:5]} ...')

In [ ]:
class TorchMLPClassifier(ClassifierMixin, BaseEstimator):
    # Sieć MLP z PyTorch jako estymator sklearn: można jej używać w Pipeline, cross_val_score,
    # permutation_importance. Early stopping na wewnętrznej części danych przekazanych do fit.
    def __init__(self, hidden=(64, 32), dropout=0.2, lr=1e-3, weight_decay=1e-4, batch_size=32,
                 max_epoki=EPOKI_MAX, cierpliwosc=CIERPLIWOSC, val_frac=0.15, random_state=SEED):
        # sklearn wymaga, by __init__ tylko zapisywał parametry (bez logiki) - inaczej clone() nie zadziała
        self.hidden = hidden; self.dropout = dropout; self.lr = lr; self.weight_decay = weight_decay
        self.batch_size = batch_size; self.max_epoki = max_epoki; self.cierpliwosc = cierpliwosc
        self.val_frac = val_frac; self.random_state = random_state

    def fit(self, X, y):
        X = np.asarray(X, dtype=np.float32); y = np.asarray(y).astype(int)
        self.classes_ = np.unique(y)                      # wymagane przez sklearn (np. scoring='roc_auc')
        ustaw_seed(self.random_state)                     # powtarzalność (uwaga: ustawia globalne ziarna)
        # Wewnętrzny podział TYLKO z danych treningowych -> early stopping bez wycieku
        X_tr, X_es, y_tr, y_es = train_test_split(X, y, test_size=self.val_frac, stratify=y,
                                                  random_state=self.random_state)
        tl = DataLoader(TabularDataset(X_tr, y_tr), batch_size=self.batch_size, shuffle=True)
        vl = DataLoader(TabularDataset(X_es, y_es), batch_size=1024)
        self.model_ = MLP(X.shape[1], self.hidden, self.dropout).to(DEVICE)
        self.model_, self.historia_ = trenuj(self.model_, tl, vl, nn.BCEWithLogitsLoss(), lr=self.lr,
                                             weight_decay=self.weight_decay, max_epoki=self.max_epoki,
                                             cierpliwosc=self.cierpliwosc, verbose=False)
        return self

    def predict_proba(self, X):
        p = proba_torch(self.model_, np.asarray(X, dtype=np.float32))
        return np.column_stack([1 - p, p])                # kolumny w kolejności self.classes_

    def predict(self, X):
        return self.classes_[(self.predict_proba(X)[:, 1] >= 0.5).astype(int)]

# Szybki test: sieć w pipeline ze skalerem, dopasowanie na train, ocena na walidacji
pipe_mlp = make_pipeline(StandardScaler(), TorchMLPClassifier())
pipe_mlp.fit(X_train, y_train)
print('Pipeline(Scaler, TorchMLP) - ROC-AUC na walidacji:',
      round(roc_auc_score(y_val, pipe_mlp.predict_proba(X_val)[:, 1]), 4))

## 6. Walidacja krzyżowa i niepewność wyników

### 6.1. Dlaczego jeden podział nie wystarcza

Wynik na jednym zbiorze walidacyjnym (114 obserwacji) jest **zmienną losową**: inny podział, inny seed - inny wynik. Walidacja krzyżowa $k$-krotna dzieli część deweloperską na $k$ foldów; każdy raz jest testem, pozostałe są treningiem. Otrzymujemy $k$ wyników $a_1,\dots,a_k$, wykorzystujemy wszystkie dane do oceny i możemy oszacować rozrzut. Powtórzona CV (*repeated k-fold*) uśrednia dodatkowo losowość podziału.

### 6.2. Przedział niepewności - i pułapka niezależności

Naiwny przedział ufności 95% dla średniej z $J$ wyników:
$$\bar a \pm t_{J-1,\,0.975}\,\frac{s}{\sqrt{J}}.$$

Zakłada on niezależność wyników, a **wyniki foldów nie są niezależne** - zbiory treningowe w różnych foldach pokrywają się w $(k-2)/(k-1)$. Bengio i Grandvalet (*JMLR*, 2004) pokazali, że nie istnieje nieobciążony estymator wariancji CV $k$-krotnej; naiwny wzór **zaniża** niepewność. Praktyczna poprawka Nadeau i Bengio (*Machine Learning*, 2003), stosowana też w testach porównujących modele (Bouckaert i Frank, 2004):
$$\hat\sigma^2_{\text{kor}} = \left(\frac{1}{J} + \frac{n_{\text{test}}}{n_{\text{train}}}\right) s^2, \qquad \bar a \pm t_{J-1,\,0.975}\,\hat\sigma_{\text{kor}}.$$

Dla 5-fold CV: $1/5 + 1/4 = 0{,}45$ zamiast $0{,}2$ - przedział szerszy o ok. 50%. To heurystyka, ale znacznie uczciwsza niż wzór naiwny.

**Porównywanie dwóch modeli:** liczmy je na **tych samych foldach** i analizujmy **różnice parami** $d_j = a^{(A)}_j - a^{(B)}_j$ - zmienność wynikająca z "trudności" foldu się wtedy znosi.

### 6.3. Bootstrap dla zbioru testowego

Dla jednego zbioru testowego niepewność metryki szacujemy bootstrapem: losujemy $B$ razy $n$ obserwacji testowych ze zwracaniem, liczymy metrykę, bierzemy percentyle 2,5% i 97,5%. Ten przedział opisuje niepewność wynikającą ze **skończonego testu** (nie uwzględnia zmienności treningu).

In [ ]:
# Demonstracja: powtórzona stratyfikowana CV sieci w pipeline (część deweloperska, test nietknięty)
cv_rep = RepeatedStratifiedKFold(n_splits=N_FOLDS, n_repeats=N_POWT, random_state=SEED)
t0 = time.time()
wyn_mlp_cv = cross_validate(make_pipeline(StandardScaler(), TorchMLPClassifier()), X_dev, y_dev, cv=cv_rep,
                            scoring=['roc_auc', 'average_precision', 'balanced_accuracy'])
print(f'Czas: {time.time() - t0:.1f} s, liczba dopasowań: {len(wyn_mlp_cv["test_roc_auc"])}')
for m in ['roc_auc', 'average_precision', 'balanced_accuracy']:
    a = wyn_mlp_cv['test_' + m]
    print(f'{m:18s}: {a.mean():.4f} +/- {a.std(ddof=1):.4f} (SD)')

In [ ]:
# Demonstracja: bootstrap 95% CI dla ROC-AUC MLP na zbiorze testowym
def bootstrap_ci(y, p, metryka, B=N_BOOT, seed=SEED):
    g = np.random.default_rng(seed); y = np.asarray(y); wyn = []
    for _ in range(B):
        idx = g.integers(0, len(y), len(y))           # losowanie ze zwracaniem
        if len(np.unique(y[idx])) < 2:                # AUC wymaga obu klas w próbie
            continue
        wyn.append(metryka(y[idx], p[idx]))
    return np.percentile(wyn, [2.5, 97.5]), np.array(wyn)

(lo, hi), boot = bootstrap_ci(y_test, p_test['MLP'], roc_auc_score)
print(f'MLP test ROC-AUC = {roc_auc_score(y_test, p_test["MLP"]):.4f}, 95% CI bootstrap: [{lo:.4f}, {hi:.4f}]')
plt.figure(figsize=(6, 3.5))
plt.hist(boot, bins=40, color='slategray')
plt.axvline(lo, c='r', ls='--', label='2.5%'); plt.axvline(hi, c='r', ls='--', label='97.5%')
plt.xlabel('ROC-AUC w próbie bootstrap'); plt.ylabel('Liczność'); plt.title('Rozkład bootstrap ROC-AUC (test)')
plt.legend(); plt.tight_layout(); plt.show()

### Zadanie 6.1 - ręczna CV sieci i porównanie z regresją logistyczną (ok. 12 min)

Bez używania `Pipeline` i `cross_validate` (żeby zobaczyć, co dzieje się "w środku"):
1. Dla każdego foldu `StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)` na `X_dev`, `y_dev`:
   - dopasuj `StandardScaler` **tylko na części treningowej foldu**,
   - wydziel z części treningowej 15% na early stopping (stratyfikacja),
   - wytrenuj `MLP(30)` funkcją `trenuj` (`verbose=False`) i policz ROC-AUC na foldzie testowym,
   - na **tych samych** foldach dopasuj `make_pipeline(StandardScaler(), LogisticRegression(max_iter=5000))` i policz ROC-AUC.
2. Dla MLP podaj średnią, przedział naiwny i skorygowany (Nadeau-Bengio).
3. Dla różnic $d_j$ = AUC(MLP) - AUC(LogReg) podaj średnią i skorygowany 95% CI. Czy przedział zawiera 0? Sformułuj wniosek jako weryfikację hipotezy (UGŁ_U01).

**Kryteria sukcesu:** tabela wyników per fold, trzy przedziały, wniosek w jednym zdaniu. Przedział skorygowany jest szerszy od naiwnego. Jeśli górna granica przekracza 1, to sygnał, że przybliżenie rozkładem t jest przy wynikach bliskich 1 zgrubne - raportuj wtedy przedział obcięty do [0, 1] i skomentuj to.

**Wskazówka:** `stats.t.ppf(0.975, J - 1)`; $n_{test}/n_{train} \approx 1/(k-1)$ - możesz policzyć dokładnie z długości indeksów.

In [ ]:
def przedzialy(a, n_test, n_train, poziom=0.95):
    # Zwraca (średnia, CI naiwny, CI skorygowany Nadeau-Bengio)
    a = np.asarray(a); J = len(a); s2 = a.var(ddof=1); tq = stats.t.ppf(0.5 + poziom / 2, J - 1)
    naiw = tq * math.sqrt(s2 / J)
    kor = tq * math.sqrt((1 / J + n_test / n_train) * s2)
    return a.mean(), (a.mean() - naiw, a.mean() + naiw), (a.mean() - kor, a.mean() + kor)

# TODO: uzupełnij (patrz polecenie powyżej)

## 7. Interpretowalność: permutation importance

**Idea** (Breiman, 2001; Fisher, Rudin, Dominici, 2019): ważność cechy $j$ to spadek jakości modelu po losowym przetasowaniu wartości tej cechy w zbiorze ocenianym:
$$I_j = s(f; X, y) - \frac{1}{R}\sum_{r=1}^{R} s\big(f; X^{(\pi_r, j)}, y\big).$$
Permutacja niszczy związek cechy z celem, zachowując jej rozkład brzegowy. Metoda jest agnostyczna względem modelu - działa tak samo dla sieci, lasu i regresji.

**Zasady i pułapki:**
- licz na zbiorze **wstrzymanym** (walidacja / test), nie na treningowym - na treningowym ważność pokazuje, co model *zapamiętał*,
- ważność dotyczy **modelu**, nie zjawiska - nie jest miarą przyczynowości,
- **skorelowane cechy** dzielą się ważnością: w Breast Cancer promień, obwód i pole są skorelowane w stopniu $r > 0{,}99$; po permutacji jednej z nich model "korzysta" z pozostałych, więc każda z osobna wydaje się mało ważna. Permutacja tworzy też nierealistyczne kombinacje (np. mały promień i duże pole) - model ocenia wtedy punkty spoza rozkładu danych,
- przy małym zbiorze ocenianym podawaj rozrzut z $R$ powtórzeń.

### Zadanie 7.1 - permutation importance dla MLP i HistGB (ok. 8 min)

1. Policz `permutation_importance` na **teście** (`X_test`, `y_test`) dla `pipe_mlp` (pipeline skaler + sieć, dopasowany w sekcji 5.9) oraz dla `modele_klas['HistGB']`, z `scoring='roc_auc'`, `n_repeats=N_PERM`, `random_state=SEED`.
2. Dla każdego modelu narysuj poziomy wykres słupkowy (`barh`) średniej ważności z odchyleniem standardowym jako słupkiem błędu (`importances_mean`, `importances_std`) dla 10 najważniejszych cech.
3. Wypisz, które cechy są w top-10 obu modeli. Narysuj macierz korelacji (heatmapa) dla cech z top-10 MLP.

**Kryteria sukcesu:** dwa czytelne wykresy z podpisami; komentarz, jak korelacje cech wpływają na interpretację. Zwróć uwagę, że ważności przy AUC bliskim 1 są małe w wartościach bezwzględnych - dlaczego?

In [ ]:
# TODO: uzupełnij (patrz polecenie powyżej)

## 8. Strojenie hiperparametrów: random search bez podglądania testu

**Grid vs random search.** Bergstra i Bengio (*JMLR*, 2012) pokazali, że zwykle tylko kilka hiperparametrów ma istotny wpływ. Przy siatce 5 x 5 testujemy tylko 5 różnych wartości każdego z nich; przy 25 losowych punktach - 25 różnych wartości każdego. Random search przy tym samym budżecie lepiej eksploruje ważne wymiary.

**Rozkłady:** współczynnik uczenia i weight decay losujemy z rozkładu **log-jednostajnego** (`scipy.stats.loguniform`) - różnica między $10^{-4}$ a $10^{-3}$ jest tak samo istotna jak między $10^{-3}$ a $10^{-2}$.

**Metody zaawansowane:** optymalizacja bayesowska (Optuna - TPE, KerasTuner), Hyperband / successive halving (`HalvingRandomSearchCV`) - wczesne odrzucanie słabych konfiguracji.

**Rzetelność:**
- każda konfiguracja oceniana walidacją krzyżową na części **deweloperskiej** (cały pipeline ze skalerem w środku),
- wynik CV najlepszej konfiguracji jest **optymistyczny** (winner's curse, sekcja 5.8) - nie raportujemy go jako końcowego,
- najlepszą konfigurację trenujemy na całej części deweloperskiej i oceniamy **raz** na teście (lub stosujemy zagnieżdżoną CV - zadanie dodatkowe).

### Zadanie 8.1 - random search dla `TorchMLPClassifier` (ok. 10 min)

1. Zdefiniuj przestrzeń: `hidden` z `[(32,), (64, 32), (128, 64)]`, `dropout` z `[0.0, 0.1, 0.3]`, `lr ~ loguniform(1e-4, 1e-2)`, `weight_decay ~ loguniform(1e-6, 1e-2)`, `batch_size` z `[16, 32, 64]`.
2. Dla `N_ITER` konfiguracji z `ParameterSampler(..., random_state=SEED)` policz średnie ROC-AUC z `cross_val_score` (`StratifiedKFold(N_FOLDS, shuffle=True, random_state=SEED)`) dla `make_pipeline(StandardScaler(), TorchMLPClassifier(**params))` na `X_dev`, `y_dev`. Zapisz wyniki w DataFrame posortowanym malejąco (`wyniki_rs`).
3. Najlepszą konfigurację dopasuj na całym `X_dev` i oceń **raz** na `X_test` (ROC-AUC + bootstrap CI z sekcji 6).
4. Porównaj najlepszy wynik CV z wynikiem testowym i skomentuj.

**Kryteria sukcesu:** tabela konfiguracji, jeden wynik testowy z przedziałem. Wniosek: czy strojenie poprawiło wynik względem konfiguracji domyślnej (sekcja 6) w sposób wykraczający poza niepewność?

In [ ]:
# TODO: uzupełnij (patrz polecenie powyżej)

## Pytania kontrolne

1. Dlaczego w klasyfikacji binarnej w PyTorch używamy `BCEWithLogitsLoss` zamiast `Sigmoid` + `BCELoss`? Co zwraca ostatnia warstwa sieci?
2. Co się stanie, jeśli w early stopping zapiszesz `best_state = model.state_dict()` bez `deepcopy`?
3. Dlaczego próg decyzyjny wybieramy na walidacji, a nie na teście? Czy wybór progu zmienia ROC-AUC?
4. Model ma ROC-AUC = 0,95. Czy to znaczy, że jego prawdopodobieństwa są dobrze skalibrowane? Jak to sprawdzić?
5. Podaj trzy przykłady wycieku danych, których **nie** usunie użycie `sklearn.Pipeline`.
6. W selekcji cech na czystym szumie procedura z wyciekiem dała accuracy ok. 0,9. Dlaczego, i jak można by tę wadę wykryć bez znajomości prawdy?
7. Dlaczego naiwny przedział ufności dla średniej z wyników foldów CV jest zbyt wąski? Na czym polega poprawka Nadeau-Bengio?
8. Permutation importance pokazuje, że `mean radius` jest prawie nieistotna, choć klinicznie wielkość guza ma znaczenie. Jak to wyjaśnić?

## Podsumowanie - co warto zapamiętać

- **PyTorch:** `Dataset` + `DataLoader` + `nn.Module` + ręczna pętla (`train()`/`eval()`, `zero_grad`, `backward`, `step`, `no_grad`). Logity + `BCEWithLogitsLoss`; tensory `float32`; etykiety `[n, 1]`.
- **Early stopping** = wybór epoki na walidacji + przywrócenie najlepszych wag (`deepcopy`!). Zapisujemy `state_dict` **oraz** przetwarzanie (skaler) i próg.
- **Baseline jest obowiązkowy.** Na danych tabelarycznych GBDT często dorównuje MLP lub go przewyższa.
- **Metryki:** ROC-AUC (ranking), PR-AUC (przy niezbalansowaniu), Brier/kalibracja (jakość prawdopodobieństw), metryki progowe zależne od decyzji. Próg dobieramy do kosztów błędów na walidacji.
- **Regresja:** MAE, RMSE, $R^2$ + analiza reszt (struktura, heteroskedastyczność, artefakty danych jak ucięcie celu).
- **Wyciek danych:** walidujemy *procedurę*, nie model. Wszystko, co ma `fit`, idzie do `Pipeline` i jest dopasowywane w każdym foldzie. Podział według jednostki wdrożenia (pacjentka, ośrodek, czas). Cechy tylko z informacją dostępną w chwili predykcji. Test używany raz.
- **Niepewność:** CV powtórzona, porównania parami na tych samych foldach, przedziały skorygowane (Nadeau-Bengio), bootstrap dla testu. Różnica bez przedziału nie jest wynikiem.
- **Permutation importance** na danych wstrzymanych; ważność dotyczy modelu, nie przyczynowości; uwaga na korelacje cech.
- **Random search** z rozkładami log-jednostajnymi; najlepszy wynik CV jest optymistyczny - rzetelną ocenę daje test lub zagnieżdżona CV.

## Do sprawozdania końcowego

Z tego laboratorium umieść w sprawozdaniu (zwięźle, z wykresami i tabelami):
1. **Tabelę porównawczą** MLP vs LogReg / RF / HistGB dla Breast Cancer (ROC-AUC, PR-AUC, Brier) oraz dla California Housing (MAE, RMSE, $R^2$) - z informacją, na jakim zbiorze liczono metryki.
2. **Wynik zadania 6.1**: AUC MLP z przedziałem skorygowanym i test różnicy MLP - LogReg, sformułowany jako weryfikacja hipotezy (UGŁ_U01).
3. **Wykres z zadania 5.1** (wyciek przy selekcji cech) lub **5.2** (grupy) z 2-3 zdaniami wyjaśnienia mechanizmu.
4. **Dobór progu** (3.1) i krzywe kalibracji (3.2) - jako przykład usprawnienia rozwiązania (UGŁ_U02).
5. **Analizę reszt** regresji (4.1) z interpretacją ucięcia celu.
6. Krótki opis **pipeline'u** zastosowanego w mini-projekcie: gdzie dzielisz dane, jakie kroki są w `Pipeline`, jaka jest jednostka podziału i dlaczego nie ma w nim wycieku.

## Zadanie dodatkowe (dla chętnych) - zagnieżdżona walidacja krzyżowa i optymizm selekcji modelu

Wróć do czystego szumu z sekcji 5.4 (prawdziwa dokładność = 0,5), ale tym razem **bez** wycieku przy selekcji cech - wszystko jest w `Pipeline`. Zamiast tego stroimy hiperparametry:
- `select__k` z `[5, 10, 20, 50, 100]` i `clf__C` z `np.logspace(-3, 2, 6)` (30 konfiguracji), `GridSearchCV` z 5-fold CV.

1. Podaj `best_score_` z `GridSearchCV` dopasowanego na całym `X_szum` - to wynik "najlepszej konfiguracji", który często trafia do publikacji.
2. Oblicz **zagnieżdżoną CV**: `cross_val_score(GridSearchCV(...), X_szum, y_szum, cv=StratifiedKFold(5, shuffle=True, random_state=1))` - pętla zewnętrzna ocenia **całą procedurę strojenia**.
3. Porównaj oba wyniki z prawdą (0,5). Powtórz dla 3 różnych ziaren danych, jeśli czas pozwala.

**Oczekiwany wniosek:** `best_score_` jest zawyżone (selekcja maksimum z 30 zaszumionych estymat), zagnieżdżona CV jest bliska 0,5. To wyciek typu 7 - bez błędu w kodzie przetwarzania.

In [ ]:
# TODO: uzupełnij (patrz polecenie powyżej)